In [1]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import AsyncRedisSaver
from pprint import pprint   

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import (
    REDIS_URL,
    install_safe_pending_sends_loader,
)

install_safe_pending_sends_loader()

MODE = "own_same_thread"
parent_thread_id = f"ckpt-mode-{MODE}"
parent_config = RunnableConfig(configurable={"thread_id": parent_thread_id})

async with AsyncRedisSaver.from_conn_string(REDIS_URL) as ch:
    await ch.asetup()
    chp_list = [t async for t in ch.alist(parent_config)]

print(f"alist returned {len(chp_list)} CheckpointTuple(s) for parent_config={parent_config}\n")
pprint(chp_list[-6], indent=2)
# Per-tuple summary, oldest -> newest
for i, tup in enumerate(reversed(chp_list)):
    cfg = tup.config["configurable"]
    msgs = tup.checkpoint.get("channel_values", {}).get("messages", [])
    print(
        f"[{i}] thread={cfg['thread_id']}  "
        f"ns={cfg.get('checkpoint_ns', '')!r}  "
        f"ckpt_id={cfg['checkpoint_id'][-15:]}  "
        f"parent_ckpt_id={tup.metadata.get('parents', {}).get("", [])[-15:]}  "
        f"step={tup.metadata.get('step'):<3} "
        f"source={tup.metadata.get('source'):<8} "
        f"msgs={len(msgs)}"
    )

# Group by location so it's easy to see which checkpoints belong to parent vs subagent
print("\n--- distinct (thread_id, checkpoint_ns) seen ---")
by_loc: dict[tuple[str, str], int] = {}
for tup in chp_list:
    key = (
            tup.config["configurable"]["thread_id"],
            tup.config["configurable"].get("checkpoint_ns", ""),
            tup.config["configurable"].get("checkpoint_id", ""),
        )
    by_loc[key] = by_loc.get(key, 0) + 1
for (t, n, ch_id), c in sorted(by_loc.items()):
    print(f"  thread={t}  ns={n!r}  ckpt_id={ch_id[-10:]}  count={c}")

# Peek at one full tuple structure
if chp_list:
    head = chp_list[0]
    print("\n--- head CheckpointTuple structure ---")
    print(f"  config.configurable    : {head.config['configurable']}")
    print(f"  checkpoint top-level   : {list(head.checkpoint.keys())}")
    print(f"  metadata               : {dict(head.metadata or {})}")
    print(f"  channel_values keys    : {list(head.checkpoint.get('channel_values', {}).keys())}")
    print(f"  parent_config          : {head.parent_config}")
    print(f"  pending_writes count   : {len(head.pending_writes or [])}")
    if head.pending_writes:
        print(f"  pending_writes[0]      : {head.pending_writes[0]}")


/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/agentic_patterns/subagent_pattern/checkpointer_modes_experiment.py:28: UserWarning: WARNING! cache_control is not default parameter.
                cache_control was transferred to model_kwargs.
                Please confirm that cache_control is what you intended.
  from utils import MODELS, Model


alist returned 15 CheckpointTuple(s) for parent_config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread'}}

CheckpointTuple(config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread', 'checkpoint_ns': 'tools:7857bbf2-264a-9928-667b-6e8b2f84d7de', 'checkpoint_id': '1f1483ee-e3bb-64ec-bfff-89047ac02233'}}, checkpoint={'type': 'json', 'v': 4, 'ts': '2026-05-05T04:57:34.725847+00:00', 'id': '1f1483ee-e3bb-64ec-bfff-89047ac02233', 'channel_values': {'__start__': {'messages': [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]}}, 'channel_versions': {'__start__': '00000000000000000000000000000001.0.7794106066210351'}, 'versions_seen': {'__input__': {}}, 'updated_channels': ['__start__'], 'pending_sends': []}, metadata={'source': 'input', 'step': -1, 'parents': {'': '1f1483ee-e38a-6c78-8001-6fbd457880a4'}}, parent_config=None, pending_writes=[('5da665bb-9b92-a782-e6ce-8b1cec4db789', 'messages', [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]), ('5

/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/utils/__init__.py:3: UserWarning: WARNING! thinking is not default parameter.
                thinking was transferred to model_kwargs.
                Please confirm that thinking is what you intended.
  from utils.llms import MODELS, Model
/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/utils/__init__.py:3: UserWarning: WARNING! cache_control is not default parameter.
                cache_control was transferred to model_kwargs.
                Please confirm that cache_control is what you intended.
  from utils.llms import MODELS, Model
/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/utils/__init__.py:3: UserWarning: WARNING! output_config is not default parameter.
                output_config was transferred to model_kwargs.
                Please confirm that output_config is what you intended.
  from utils.llms import MODELS, Model


In [2]:
import base64
from typing import Any

import orjson
from redis.asyncio import Redis

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import REDIS_URL

MODE = "own_same_thread"
thread_id = f"ckpt-mode-{MODE}"


def try_decode_blob(b64: str) -> tuple[bool, str | bytes | None]:
    try:
        raw = base64.b64decode(b64)
    except Exception as e:
        return False, f"base64: {e}"
    try:
        orjson.loads(raw)
    except Exception:
        return False, raw
    return True, raw


def walk_doc(node: Any, path: str, out: list[dict]) -> None:
    if isinstance(node, dict):
        if "blob" in node and isinstance(node["blob"], str):
            ok, payload = try_decode_blob(node["blob"])
            if not ok:
                out.append({"path": f"{path}.blob", "type": node.get("type"),
                            "channel": node.get("channel"), "err_or_raw": payload})
        if "__bytes__" in node and isinstance(node["__bytes__"], str):
            ok, payload = try_decode_blob(node["__bytes__"])
            if not ok:
                out.append({"path": f"{path}.__bytes__", "type": "bytes-marker",
                            "channel": None, "err_or_raw": payload})
        for k, v in node.items():
            walk_doc(v, f"{path}.{k}", out)
    elif isinstance(node, list):
        for i, v in enumerate(node):
            walk_doc(v, f"{path}[{i}]", out)


PREFIXES = ["checkpoint", "checkpoint_write", "checkpoint_latest"]
totals: dict[str, dict[str, int]] = {p: {"docs": 0, "json": 0, "bad": 0} for p in PREFIXES}
all_bad: list[dict] = []
non_json: dict[str, list[str]] = {p: [] for p in PREFIXES}

async with Redis.from_url(REDIS_URL, decode_responses=False) as r:
    for prefix in PREFIXES:
        keys = [k async for k in r.scan_iter(match=f"{prefix}:{thread_id}*".encode())]
        totals[prefix]["docs"] = len(keys)
        for k in keys:
            ktype = (await r.type(k)).decode()
            if ktype != "ReJSON-RL":
                non_json[prefix].append(f"{k.decode()}  (type={ktype})")
                continue
            totals[prefix]["json"] += 1
            doc = await r.json().get(k)
            if not isinstance(doc, (dict, list)):
                continue
            findings: list[dict] = []
            walk_doc(doc, "", findings)
            if findings:
                totals[prefix]["bad"] += 1
                for f in findings:
                    f["key"] = k.decode()
                    f["prefix"] = prefix
                all_bad.extend(findings)

print(f"{'prefix':<22} {'docs':>6} {'json':>6} {'bad':>6}")
for p, t in totals.items():
    print(f"{p:<22} {t['docs']:>6} {t['json']:>6} {t['bad']:>6}")

for p, lst in non_json.items():
    if lst:
        print(f"\nnon-JSON keys under {p}:")
        for s in lst[:5]:
            print(f"  {s}")

print(f"\ntotal bad blobs: {len(all_bad)}\n")
for entry in all_bad[:6]:
    raw = entry["err_or_raw"]
    print(f"BAD {entry['prefix']}{entry['path']}  type={entry.get('type')}  channel={entry.get('channel')}")
    print(f"  key: {entry['key']}")
    if isinstance(raw, bytes):
        print(f"  len: {len(raw)}")
        print(f"  head: {raw[:200]!r}")
        print(f"  tail: {raw[-120:]!r}")
    else:
        print(f"  err: {raw}")
    print()


prefix                   docs   json    bad
checkpoint                 15     15      0
checkpoint_write           24     24      7
checkpoint_latest           3      0      0

non-JSON keys under checkpoint_latest:
  checkpoint_latest:ckpt-mode-own_same_thread:tools:7857bbf2-264a-9928-667b-6e8b2f84d7de  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:1cdfbfa5-8898-6599-37d7-3808745e2a99  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:__empty__  (type=string)

total bad blobs: 7

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:__empty__:1f1483ee-e38a-6c78-8001-6fbd457880a4:1cdfbfa5-8898-6599-37d7-3808745e2a99:1
  len: 0
  head: b''
  tail: b''

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:tools:1cdfbfa5-8898-6599-37d7-3808745e2a99:1f1483ee-fa33-6470-8001-7e81c27ea995:de11b092-f38f-16cb-ee32-af6ba7cd9cb9:1
  len: 0
  head: b''


In [3]:
async with AsyncRedisSaver.from_conn_string(REDIS_URL) as ch:
    await ch.asetup()
    config = RunnableConfig(configurable={"thread_id": "dbt Agent-1"})
    chp_history = [t async for t in ch.alist(config)]

In [17]:
from langchain_core.messages.utils import count_tokens_approximately
last_tup = chp_history[0]if chp_history else None

chkpt = last_tup.checkpoint if last_tup else None
messages  = chkpt.get("channel_values", {}).get("messages", [])

print(f"last checkpoint has {len(messages)} messages")
approximate_tokens = count_tokens_approximately(messages, chars_per_token=3)
print(f"approximate token count: {approximate_tokens}") 



last checkpoint has 415 messages
approximate token count: 196628


In [18]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
compaction_messages = messages.copy()
total_messages = len(compaction_messages)
def get_message_counts(messages):
    counts = {"human": 0, "ai": 0, "tool": 0}
    for msg in messages:
        if isinstance(msg, HumanMessage):
            counts["human"] += 1
        elif isinstance(msg, AIMessage):
            counts["ai"] += 1
        elif isinstance(msg, ToolMessage):
            counts["tool"] += 1
    return counts
counts = get_message_counts(compaction_messages)
print(f"message counts: {counts}")

message counts: {'human': 12, 'ai': 200, 'tool': 203}


In [44]:
tools_by_type = {
"write_tools" : ["Write", "WriteArtifact",],
"edit_tools" : ["Edit", "EditArtifact",],
"read_tools" : ["Read", "ReadArtifact",],
"misc" : ["Todos", "Skill"],
"bash" : ["bash", "Grep"],
"sf_tools" : [
    "MANAGE_SNOWFLAKE_OBJECTS", "PUBLISH_SEMANTIC_VIEW", "QUERY_SNOWFLAKE", "DEPLOY_CORTEX_AGENT"
],
}

def print_pruning_summary(tool_type, pruned_counts, pruned_token_estimate):
    print(f"\n--- pruning {tool_type} call messages ---")
    print(f"after pruning {tool_type} call messages: {pruned_counts}, token estimate: {pruned_token_estimate}")
    print(
        f"orignal vs pruned estimate: {approximate_tokens} -> {pruned_token_estimate}"
        f"  ({approximate_tokens - pruned_token_estimate} tokens removed)"
    )

def prune_tool_calls(messages, tool_names = []):
    new_messages = []
    for msg in messages:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                if any(call.get("name") in tool_names for call in msg.tool_calls):
                    continue
        new_messages.append(msg)
    return new_messages

def prune_tool_messages(messages):
    new_messages = []
    for msg in messages:
        if isinstance(msg, ToolMessage):
            continue
        new_messages.append(msg)
    return new_messages

tools_pruning = {}
for tool_type, tool_names in tools_by_type.items():
    pruned = prune_tool_calls(compaction_messages, tool_names=tool_names)
    pruned_counts = get_message_counts(pruned)
    pruned_token_estimate = count_tokens_approximately(pruned, chars_per_token=3)
    tools_pruning[tool_type] = {
        "counts": pruned_counts,
        "token_estimate": pruned_token_estimate,
        "tokens_removed": approximate_tokens - pruned_token_estimate,
    }
    print_pruning_summary(tool_type, pruned_counts, pruned_token_estimate)

prune_tool_results = prune_tool_messages(compaction_messages)
pruned_counts = get_message_counts(prune_tool_results)
pruned_token_estimate = count_tokens_approximately(prune_tool_results, chars_per_token=3)
print_pruning_summary("all tool messages", pruned_counts, pruned_token_estimate)


--- pruning write_tools call messages ---
after pruning write_tools call messages: {'human': 12, 'ai': 170, 'tool': 203}, token estimate: 169062
orignal vs pruned estimate: 196628 -> 169062  (27566 tokens removed)

--- pruning edit_tools call messages ---
after pruning edit_tools call messages: {'human': 12, 'ai': 134, 'tool': 203}, token estimate: 178143
orignal vs pruned estimate: 196628 -> 178143  (18485 tokens removed)

--- pruning read_tools call messages ---
after pruning read_tools call messages: {'human': 12, 'ai': 190, 'tool': 203}, token estimate: 194468
orignal vs pruned estimate: 196628 -> 194468  (2160 tokens removed)

--- pruning misc call messages ---
after pruning misc call messages: {'human': 12, 'ai': 186, 'tool': 203}, token estimate: 190942
orignal vs pruned estimate: 196628 -> 190942  (5686 tokens removed)

--- pruning bash call messages ---
after pruning bash call messages: {'human': 12, 'ai': 154, 'tool': 203}, token estimate: 187684
orignal vs pruned estimate: 

In [45]:
read_tools = tools_by_type["read_tools"]
def tool_messages_per_type(messages, tool_names = []):
    tool_messages = []
    tool_ids = set()
    for msg in messages:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                if any(call.get("name") in tool_names for call in msg.tool_calls):
                    for call in msg.tool_calls:
                        if call.get("name") in tool_names:
                            tool_ids.add(call.get("id"))
    for id in tool_ids:
        for msg in messages:
            if isinstance(msg, ToolMessage) and msg.tool_call_id == id:
                tool_messages.append(msg)
                    
    return tool_messages

def tool_results_per_type_report(messages, tool_names = []):
    
    tool_msgs = tool_messages_per_type(messages, tool_names=tool_names)
    tool_msgs_token_estimate = count_tokens_approximately(tool_msgs, chars_per_token=3)
    tool_msgs_counts = get_message_counts(tool_msgs)

    print(f"\n--- result messages for {', '.join(tool_names)} ---")
    print(f"message counts: {tool_msgs_counts}, token estimate: {tool_msgs_token_estimate}")
    print(
        f"orignal has {tool_type} resutl messages estimate:\n" \
        f"Total tokens:{approximate_tokens} -> {tool_msgs_token_estimate}"
    )

for tool_type, tool_names in tools_by_type.items():
    print(f"\n=== report for {tool_type} ===")
    tool_results_per_type_report(messages, tool_names=tool_names)




=== report for write_tools ===

--- result messages for Write, WriteArtifact ---
message counts: {'human': 0, 'ai': 0, 'tool': 30}, token estimate: 1514
orignal has write_tools resutl messages estimate:
Total tokens:196628 -> 1514

=== report for edit_tools ===

--- result messages for Edit, EditArtifact ---
message counts: {'human': 0, 'ai': 0, 'tool': 66}, token estimate: 16093
orignal has edit_tools resutl messages estimate:
Total tokens:196628 -> 16093

=== report for read_tools ===

--- result messages for Read, ReadArtifact ---
message counts: {'human': 0, 'ai': 0, 'tool': 15}, token estimate: 33744
orignal has read_tools resutl messages estimate:
Total tokens:196628 -> 33744

=== report for misc ===

--- result messages for Todos, Skill ---
message counts: {'human': 0, 'ai': 0, 'tool': 14}, token estimate: 24410
orignal has misc resutl messages estimate:
Total tokens:196628 -> 24410

=== report for bash ===

--- result messages for bash, Grep ---
message counts: {'human': 0, 'a